In [1]:
import numpy as np
import torch
import scanpy as sc
import anndata as ad
import os
import pandas as pd
from utils.preprocess import *

In [2]:
import os
import sys
    
import matplotlib.pyplot as plt
import numpy as np
import scanpy as sc
import torch
import torchsde
from torchdyn.core import NeuralODE
from tqdm import tqdm

from torchcfm.conditional_flow_matching import *
from torchcfm.models import MLP
from torchcfm.utils import plot_trajectories, torch_wrapper
from simulate.simulate import *
from omegaconf import OmegaConf
from utils.hydra import *
from datasets.process import *
from scripts.run_model import *
from eval.eval import *

In [3]:
import matplotlib.pyplot as plt
%matplotlib inline

In [4]:
%reload_ext autoreload
%autoreload 2

In [5]:
############################################################

In [16]:
### SETTINGS ###
config = load_config()

OmegaConf.set_struct(config, False)
config.pc_dim = 100
adata = process_data(pc_dim=config.pc_dim, data="cite")
timepoints = sorted(adata.obs['timepoint'].unique().tolist())
tree = adata.uns['tree']

config.num_classes = adata.obs['cell_type'].nunique()

#######################
config.metric = "cfm"
config.finsler.use = False
config.finsler.lamb = 3.0
config.dummy_kl_weight = 0.1

config.K = 150
config.kappa = 1.5
config.classifier_max_epochs = 2
config.metric_max_epochs = 2
config.embed_max_epochs = 2000
config.flow_max_epochs = 2

project = "cite"

/home/azweig/projects/finfm/utils/lineage.py:75: ImplicitModificationWarning: Trying to modify attribute `._uns` of view, initializing view as actual.
  adata.uns['stoi'] = stoi


In [17]:
adata.obs['donor'].unique()

array([32606])

In [7]:
t_holdout_index = 1
# t_holdout_index = 2

t0 = timepoints[t_holdout_index-1]
t = timepoints[t_holdout_index]
t1 = timepoints[t_holdout_index+1]
adata = adata[adata.obs['timepoint'].isin([t0, t, t1])]

In [8]:
test_bool = adata.obs['timepoint'] == t
adata_train = adata[~test_bool]
adata_test = adata[test_bool]

In [9]:
############################################################

In [10]:
singleton_dataloader = build_singleton_dataloader(config, adata_train)
paired_dataloader = build_paired_dataloader(config, adata_train)

In [11]:

classifier_model, metric_model, embed_model, flow_model = run_full_model(config=config,
                                                                         project=project,
                                                                         singleton_dataloader=singleton_dataloader,
                                                                         paired_dataloader=paired_dataloader,
                                                                         timepoints=timepoints,
                                                                         tree=tree)

Running phase classifier:.......


wandb: Currently logged in as: az831 (az831-new-york-genome-center) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
/home/azweig/miniconda3/envs/spatialenv/lib/python3.10/site-packages/pytorch_lightning/loggers/wandb.py:397: There is a wandb run already in progress and newly created instances of `WandbLogger` will reuse this run. If this is not desired, call `wandb.finish()` before instantiating `WandbLogger`.
wandb: logging graph, to disable use `wandb.watch(log_graph=False)`
You are using a CUDA device ('NVIDIA GeForce RTX 4080 SUPER') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name           | Type           | Params | Mode 
----------------------------------------------------

epoch,▁▁▁▁▁▁▁███████
train_ce,█▇▇▆▆▅▄▄▃▃▂▂▁▁
train_kl,▂▁▁▁▁▁▂▂▃▃▅▅▇█
train_loss,█▇▇▆▆▅▄▄▃▃▂▂▁▁
trainer/global_step,▁▂▂▃▃▄▄▅▅▆▆▇▇█
epoch,1
train_ce,1.55599
train_kl,0.16201
train_loss,1.5722
trainer/global_step,13


Running phase metric:.......


...

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
/home/azweig/miniconda3/envs/spatialenv/lib/python3.10/site-packages/pytorch_lightning/loggers/wandb.py:397: There is a wandb run already in progress and newly created instances of `WandbLogger` will reuse this run. If this is not desired, call `wandb.finish()` before instantiating `WandbLogger`.
wandb: logging graph, to disable use `wandb.watch(log_graph=False)`
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name         | Type           | Params | Mode 
--------------------------------------------------------
0 | embed_net    | SimpleEmbedNet | 184 K  | train
1 | geo_net      | SinNet         | 226 K  | train
2 | metric_model | MetricNetCFM   | 0      | eval 
--------------------------------------------------------
411 K     Trainable params
0         Non-trainable params
411 K     Total params
1.645     Total estimated model params size (MB)
26        Modules in train 

epoch,▁▁▁▁▁▂▂▂▂▂▂▂▂▃▃▃▃▃▃▃▄▄▄▄▄▅▅▆▆▆▆▇▇▇▇▇████
train_loss_embed,▇▇█▇▆▆▄▄▆█▅▃▄▄▃▂▂▂▂▂▂▂▂▁▂▂▁▂▁▂▂▂▁▁▂▂▁▁▁▁
train_loss_geo,▅▇▇█▄▄▅▆▆▆▄▃▂▆▃▆▄▄▇▃▃▄▆▄▁▅▄▅▄▅▃▃▂▆▂▅▅▅▃▄
trainer/global_step,▁▁▁▁▂▂▂▂▂▂▃▃▃▃▃▃▄▄▄▄▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇▇▇██
epoch,1999
train_loss_embed,10.86756
train_loss_geo,3873.01733
trainer/global_step,1999


Running phase flow:.......


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
/home/azweig/miniconda3/envs/spatialenv/lib/python3.10/site-packages/pytorch_lightning/loggers/wandb.py:397: There is a wandb run already in progress and newly created instances of `WandbLogger` will reuse this run. If this is not desired, call `wandb.finish()` before instantiating `WandbLogger`.
wandb: logging graph, to disable use `wandb.watch(log_graph=False)`
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name        | Type              | Params | Mode 
----------------------------------------------------------
0 | flow_net    | SinNet            | 201 K  | train
1 | embed_model | EmbedNetTrainBase | 411 K  | eval 
----------------------------------------------------------
201 K     Trainable params
411 K     Non-trainable params
612 K     Total params
2.450     Total estimated model params size (MB)
13        Modules in train mode
28        Modules in eval mode
/home

epoch,▁█
train_loss,▁█
trainer/global_step,▁█
epoch,1
train_loss,86.55009
trainer/global_step,1


In [12]:
#fix the wandb.run.summary bug?
def remove_all_forward_hooks(model):
    for module in model.modules():
        module._forward_hooks.clear()

remove_all_forward_hooks(classifier_model)
remove_all_forward_hooks(metric_model)
remove_all_forward_hooks(embed_model)
remove_all_forward_hooks(flow_model)

In [13]:
print(predict(embed_model, adata, t, num_traj=6000, library="geomloss"))

23.1154727935791
